# Proposition 4.8 — Whittaker vector and GHT formula

This notebook verifies **Proposition 4.8** of

> J.-E. Bourgine, L. Cassia, A. Stoyan,
> *Generalized Macdonald functions and quantum toroidal gl(1) algebra* (2025),
> [arXiv:2508.19704](https://arxiv.org/abs/2508.19704).

## Mathematical background

### The operator $\mathsf{V}$ (eq. 4.42)

The operator `VV(x)` implements eq. (4.42): it acts on an element $x$ of
the $N$-fold tensor product of symmetric functions by applying the framing
operator, a diagonal plethystic substitution
$p_1 \mapsto p_1/t - \epsilon\,q_3^i\,\varepsilon_\emptyset$
in each factor, multiplication by a plethystic exponential in the
total variable $\sum_i X_i$, and a final framing step on the resulting
lazy power series — with $\epsilon \to -1$ substituted at the end.

### Whittaker vector (eq. 4.56–4.57)

The **Whittaker vector** `Whittaker(lam, n, shift)` computes the degree-$n$
component of the Whittaker vector associated to the multi-partition
$\boldsymbol{\lambda}$, via the recursion (4.56):

$$W_{\boldsymbol{\lambda}}^{(n)} = \sum_{\boldsymbol{\nu}} M(\boldsymbol{\lambda},\boldsymbol{\nu})
\sum_{k=0}^{n} s_{[k]}\!\left[\frac{1-t}{1-q}\Bigl(\sum_a u_a\,\varepsilon_{\lambda^{(a)}}
- \sum_{a>0} q_3\,u_a\,\varepsilon_{\nu^{(a-1)}}\Bigr) X_0\right]
\cdot W_{\boldsymbol{\nu}}^{(n-k)}$$

where the outer sum runs over $(N-1)$-tuples $\boldsymbol{\nu}$ with
$\nu^{(a)} \subseteq \lambda^{(a+1)}$ for each $a$, and
$M(\boldsymbol{\lambda}, \boldsymbol{\nu})$ is the explicit Nekrasov factor
coefficient given in eq. (4.57):

$$M(\boldsymbol{\lambda},\boldsymbol{\nu}) =
u_0^{|\boldsymbol{\lambda}|-|\boldsymbol{\nu}|}
\prod_{b<a} \frac{N_{\nu^{(b)},\lambda^{(a)}}(q_3 u_{b+1}/u_a)}{N_{\lambda^{(b)},\lambda^{(a)}}(u_b/u_a)}
\prod_{b\leq a} \frac{N_{\lambda^{(b)},\nu^{(a)}}(u_b/u_{a+1})}{N_{\nu^{(b)},\nu^{(a)}}(q_3 u_{b+1}/u_{a+1})}$$

where $N_{\lambda,\mu}(z)$ denotes the Nekrasov factor (`NekrasovJEB`).
The `shift` parameter offsets the spectral parameters
$u_a \mapsto u_{a+\text{shift}}$, enabling the recursion to track
the correct parameters at each depth.

### GHT formula (Proposition 4.8)

The proposition identifies the generalised Macdonald K-function $K_{\boldsymbol{\lambda}}$
(`GMK`) with a specific combination of Whittaker vectors:

$$K_{\boldsymbol{\lambda}} := t^{-L_0}\,\mathrm{PE}\!\left[-\epsilon\,\varepsilon_\emptyset\sum_i q_3^i\frac{\partial}{\partial X_i}\right]
\cdot\nabla(\boldsymbol{u})\cdot\tilde{P}_{\boldsymbol{\lambda}}
\overset{\text{GHT}}{=} \mathrm{PE}\!\left[\epsilon\,
\frac{1-t}{1-q}\,\varepsilon_\emptyset\sum_i X_i\right]
\cdot \nabla(\boldsymbol{u})^{-1}\cdot W_{\boldsymbol{\lambda}}$$

where $\tilde{P}_{\boldsymbol{\lambda}}$ is the GMP in the spherical normalization and both sides of $\overset{\text{GHT}}{=}$ are computed as lazy power series in the tensor product
of symmetric functions and compared up to degree $|\boldsymbol{\lambda}|$.

In [ ]:
load("../gmplib.py")

## Operator $\mathsf{V}$ (eq. 4.42)

`framing_on_LSF` is a helper that lifts `framing_on_tensor` to a lazy
symmetric function series, applying the framing operator degree by degree.
`VV(x)` then implements the full operator $\mathsf{V}$ of eq. (4.42).

In [ ]:
def framing_on_LSF(x, power=1):
    N = len(x[0].parent().tensor_factors())
    LSF = LazySymmetricFunctions(tensor([p]*N))
    return LSF(lambda n: framing_on_tensor(x[n], power), valuation=0)

def VV(x):
    N = len(x.parent().tensor_factors())
    X = [tensor([p[1] if j==i else p.one() for j in range(N)]) for i in range(N)]
    x = framing_on_tensor(x)
    x = diag_plethysm(x, [p[1]/t - r*q3^i*epsilon([]) for i in range(N)])
    x = exp( LSF(lambda n: (-1)^n*p[n](sum(X)/(1-q))/n, valuation=1) ) * x
    return framing_on_LSF(x.map_coefficients(lambda c: c.subs(r=-1)))

## Whittaker vector (eq. 4.56–4.57)

`MMM(lam, nu, shift)` computes the coefficient $M(\boldsymbol{\lambda}, \boldsymbol{\nu})$
of eq. (4.57) as a product of `NekrasovJEB` factors.

`Whittaker(lam, n, shift)` computes the degree-$n$ component of the
Whittaker vector via the recursion (4.56), bottoming out at $N=0$.

In [ ]:
def MMM(lam, nu, shift=0):
    N = len(lam)
    vu = [u[a+shift] for a in range(N)]
    res = vu[0]**sum(map(sum,lam))
    res *= prod((vu[0])**-sum(k) for k in nu)
    res *= prod(NekrasovJEB(nu[b],lam[a],q3*vu[b+1]/vu[a])/NekrasovJEB(lam[b],lam[a],vu[b]/vu[a]) for a in range(N) for b in range(a))
    res *= prod(NekrasovJEB(lam[b],nu[a],vu[b]/vu[a+1])/NekrasovJEB(nu[b],nu[a],q3*vu[b+1]/vu[a+1]) for a in range(N-1) for b in range(a+1))
    return res

def Whittaker(lam, n, shift=0):
    N = len(lam)
    vu = [u[a+shift] for a in range(N)]
    one = tensor([p.one()]*(N+shift))
    zero = tensor([p.zero()]*(N+shift))
    X = [tensor([p.one()]*shift+[p[1] if j==i else p.one() for j in range(N)]) for i in range(N)]
    if N == 0:
        return one if n == 0 else zero
    else:
        nulist = list(list(subpart(lam[a])) for a in range(1, N))
        return sum(
            MMM(lam, nu, shift) * sum(
                evalArg(s[k], (1-t)/(1-q) * (sum(vu[a]*epsilon(lam[a]) for a in range(N))
                    - sum(q3*vu[a]*epsilon(nu[a-1]) for a in range(1,N))) * X[0])
                * Whittaker(nu, n-k, shift+1)
                for k in range(n+1))
            for nu in product(*nulist))

## Verification of Proposition 4.8

`testGHT(lam)` checks the GHT formula for a given multi-partition
$\boldsymbol{\lambda}$ by computing both sides as lazy power series
and comparing up to degree $|\boldsymbol{\lambda}|$.

In [ ]:
def testGHT(lam):
    N = len(lam)
    X = [tensor([p[1] if j==i else p[[]] for j in range(N)]) for i in range(N)]
    LSF = LazySymmetricFunctions(tensor([p]*N))
    rhs = (exp(LSF(lambda n: (-1)^n*p[n]((1-t)/(1-q)*epsilon([])*sum(X))/n, valuation=1))
           * LSF(lambda n: framing_on_tensor(Whittaker(lam,n), power=-1), valuation=0))
    return (GMK(lam) - rhs).truncate(sum(map(sum,lam))) == 0

In [ ]:
all(testGHT(lam) for d in range(4) for lam in mPartitions(2, d))